# Predicting `claim_within_year`
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows, selection by mean CV ROC AUC, holdout scored once.

All features (`channel`, `device_age_years`, `price_eur`, `daily_hours`) are recorded at or before the sale/registration per `columns.csv`, so they are known at prediction time. `device_serial` is excluded (identifier).

In [ ]:
import json
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

SEED = 0
df = pd.read_csv('data.csv')
holdout_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
train_idx = np.setdiff1d(np.arange(len(df)), holdout_idx)
TARGET = 'claim_within_year'
cat = ['channel']; num = ['device_age_years', 'price_eur', 'daily_hours']
features = cat + num
X, y = df[features], df[TARGET]
print(len(train_idx), len(holdout_idx), y.iloc[train_idx].mean())

## Candidates (named before any scoring)
Preprocessing lives inside each pipeline, so it is refit within every CV fold.

In [ ]:
def pre(scale):
    return ColumnTransformer([('c', OneHotEncoder(handle_unknown='ignore'), cat),
                              ('n', StandardScaler() if scale else 'passthrough', num)])
candidates = {
    'baseline_prior': Pipeline([('p', pre(False)), ('m', DummyClassifier(strategy='prior'))]),
    'logreg_C1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=1.0, max_iter=1000))]),
    'logreg_C0.1': Pipeline([('p', pre(True)), ('m', LogisticRegression(C=0.1, max_iter=1000))]),
    'random_forest': Pipeline([('p', pre(False)), ('m', RandomForestClassifier(n_estimators=300, min_samples_leaf=20, random_state=SEED, n_jobs=-1))]),
    'hist_gb': Pipeline([('p', pre(False)), ('m', HistGradientBoostingClassifier(learning_rate=0.05, max_iter=150, max_depth=3, random_state=SEED))]),
}
list(candidates)

## Cross-validation on training rows only

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
Xtr, ytr = X.iloc[train_idx], y.iloc[train_idx]
folds = [(train_idx[a].tolist(), train_idx[b].tolist()) for a, b in skf.split(Xtr, ytr)]
cv_scores = {}
for name, model in candidates.items():
    s = []
    for tr, va in folds:
        m = clone(model).fit(X.iloc[tr], y.iloc[tr])
        s.append(float(roc_auc_score(y.iloc[va], m.predict_proba(X.iloc[va])[:, 1])))
    cv_scores[name] = s
cv_table = pd.DataFrame({k: {'mean': np.mean(v), 'std': np.std(v)} for k, v in cv_scores.items()}).T
cv_table.sort_values('mean', ascending=False)

## Selection by highest mean CV ROC AUC, then a single holdout score

In [ ]:
chosen = max(cv_scores, key=lambda k: np.mean(cv_scores[k]))
final = clone(candidates[chosen]).fit(Xtr, ytr)
holdout_auc = float(roc_auc_score(y.iloc[holdout_idx], final.predict_proba(X.iloc[holdout_idx])[:, 1]))
print('chosen:', chosen, '| mean CV AUC: %.4f' % np.mean(cv_scores[chosen]), '| holdout AUC: %.4f' % holdout_auc)

In [ ]:
evidence = {
    'features': features,
    'candidates': list(candidates),
    'cv_folds': [[tr, va] for tr, va in folds],
    'cv_scores': cv_scores,
    'chosen': chosen,
    'holdout_rows': holdout_idx.tolist(),
    'holdout_scores': {chosen: holdout_auc},
}
json.dump(evidence, open('evidence.json', 'w'))